# 📐 第六课 · 难度调整

## 本课目标

- 理解难度调整的数学公式
- 理解为什么是 2016 个区块、为什么有 4x 限制
- 亲眼看到不同算力变化场景下难度如何响应
- 了解 2021 年中国禁令对难度的真实影响

> 这是比特币"自动驾驶"机制中最精妙的部分。

## 1. 为什么需要调整难度？

比特币的目标是**每 10 分钟出一个块**。

问题：全网算力在变。如果难度不变：
- 更多矿工加入 → 出块变快（可能 1 分钟一个）→ 币发得太快
- 矿工退出 → 出块变慢（可能 1 小时一个）→ 网络堵死

解法：**让难度跟着算力走**。算力涨 → 难度涨。算力跌 → 难度跌。

就像一个恒温器：室温太高了 → 关暖气。太低了 → 开暖气。

## 2. 调整公式

每 **2016 个区块**（约 2 周）调整一次：

```
新难度 = 旧难度 × (2016 × 10分钟) / (实际出 2016 个块的时间)
```

举例：
- 如果 2016 个块花了 1 周（太快了），</b> 新难度 = 旧难度 × 2
- 如果 2016 个块花了 4 周（太慢了），    新难度 = 旧难度 × 0.5

**但有限制**：新难度不能超过旧难度的 **4 倍**，也不能低于 **1/4**。

这是防极端事件：如果全网算力一夜跌 90%，难度一次只调 1/4，
不至于让剩余矿工瞬间暴富，也不会让网络直接瘫痪。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

class DifficultyController:
    # 比特币难度调整控制器
    def __init__(self, target_block_time: float = 600.0):
        self.target_time = target_block_time     # 目标出块时间（秒）
        self.interval = 2016                      # 每 2016 块调整
        self.difficulty = 1.0                     # 初始难度
        self.block_times = deque(maxlen=self.interval)
        self.history = [1.0]  # 难度历史

    def mine_block(self, hashrate: float) -> float:
        # 模拟挖一个区块
        # 返回实际用了多少秒
        expected = (self.difficulty * 2**32) / hashrate
        actual = random.expovariate(1.0 / expected)  # 指数分布
        self.block_times.append(actual)

        if len(self.block_times) == self.interval:
            self._adjust()
            self.block_times.clear()
        return actual

    def _adjust(self):
        # 难度调整
        actual_avg = np.mean(self.block_times)
        target_total = self.target_time * self.interval
        ratio = target_total / (actual_avg * self.interval)
        # 限制在 ±4x 范围内
        ratio = max(0.25, min(4.0, ratio))
        self.difficulty *= ratio
        self.history.append(self.difficulty)

print("✅ DifficultyController 就绪")  

## 3. 三个场景模拟

下面我们模拟三种算力变化场景，观察难度如何响应：
1. **指数增长**（模拟 2009-2024 真实历史）
2. **断崖下跌**（模拟 2021 中国禁令）
3. **V 型反弹**（市场恢复）

In [ ]:
# 定义三个场景的算力变化函数
def scenario_growth(block_num):
    # 指数增长：1 GH/s 起步，每天涨 0.005%
    return 1e9 * (1.00005 ** block_num)

def scenario_crash(block_num):
    # 突然暴跌：第 5000 块后算力跌 50%
    return 1e9 if block_num < 5000 else 0.5e9

def scenario_recovery(block_num):
    # V 型：3000块后开始跌，再慢慢恢复
    if block_num < 3000:
        return 1e9
    elif block_num < 7000:
        return 1e9 * (0.2 + 0.8 * (1 - (block_num-3000)/4000))
    else:
        return 1e9 * (0.2 + 0.8 * min(1, (block_num-7000)/3000))

scenarios = [
    ("指数增长", scenario_growth),
    ("断崖下跌", scenario_crash),
    ("V型反弹", scenario_recovery),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (title, hr_fn) in zip(axes, scenarios):
    ctrl = DifficultyController()
    for i in range(20160):  # 10 个调整周期
        hr = hr_fn(i)
        ctrl.mine_block(hr)

    ax.plot(ctrl.history, "o-", markersize=2, linewidth=1, color="orange")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("调整周期 (×2016块)")
    ax.set_ylabel("难度")
    ax.grid(True, alpha=0.3)

plt.suptitle("比特币难度调整——三种场景", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/difficulty_scenarios.png", dpi=100, bbox_inches="tight")
plt.close("all")
print("✅ 图表已保存到 data/difficulty_scenarios.png")
print()
print("💡 第一张图：难度从 1 平滑增长（跟算力趋势一致）")
print("💡 第二张图：算力暴跌后，难度分步缓慢下调（因为 4x 限制）")
print("💡 第三张图：难度跟随算力 V 型走势，有滞后（正常）")  

## 4. 为什么是 2016？为什么是 4x？

**2016 块 ≈ 2 周**：
- 太短（如 100 块）→ 对短期波动过度敏感，难度跳来跳去
- 太长（如 10000 块）→ 对算力趋势响应太慢
- 2 周是经验上很合适的平衡点

**4x 限制**：
- 防极端事件
- 2021 年 5 月中国禁令：全网算力一度跌约 50%
- 难度连续 4 个调整周期才完全"消化"这个跌幅
- 如果没有 4x 限制，难度可能急剧崩塌

代价是恢复也慢——但这正是比特币"稳"的体现。

## 5. 真实难度历史

```
2009年 1月: 难度 = 1
2010年      : 难度 ~100       (GPU 挖矿)
2013年      : 难度 ~500,000   (ASIC 降临)
2017年      : 难度 ~1万亿      (牛市矿潮)
2021年 5月 : 难度 ~25万亿 → 14万亿 (中国禁令，跌42%)
2024年      : 难度 ~70万亿     (历史新高)
```

15 年，难度增长了 **70 万亿倍**。

这条曲线就是比特币的发展史——每一次技术升级、每一次牛市、每一次危机，
都写在难度曲线上。

## 本课小结

| 要点 | 记住 |
|------|------|
| 调整公式 | 新难度 = 旧难度 × (目标时间/实际时间) |
| 调整周期 | 每 2016 个块 = 约 2 周 |
| 4x 限制 | 防极端事件，代价是恢复慢 |
| 难度跟算力走 | 不跟价格走——算力=安全预算 |
| 历史见证 | 难度曲线 = 比特币发展史 |

**下一课**：挖矿经济学——矿工怎么赚钱，51% 攻击为什么没人做。